# Claude 3와 MongoDB로 RAG 시스템 만들기


이 튜토리얼에서는 벤처캐피털 기술 애널리스트 역할을 하도록 프롬프트된 챗봇을 구현합니다. 이 챗봇은 기술 뉴스 기사 모음을 지식 원천으로 삼는 단순한 RAG 시스템입니다.
이 노트북에서 다루는 내용은 다음과 같습니다.

1. 필요한 라이브러리 설치부터 MongoDB 데이터베이스 설정까지, 개발 환경을 갖추는 전체 과정을 따라갑니다.
2. 벡터 검색 인덱스 생성과 데이터 적재·질의 처리를 위한 데이터 준비 등 효율적인 데이터 처리 방법을 배웁니다.
3. 데이터베이스에서 가져온 맥락 정보를 바탕으로 정확한 응답을 생성하도록 RAG 시스템 안에서 Claude 3 모델을 활용하는 방법을 이해합니다.


다음이 필요합니다.
- Claude API 키
- VoyageAI API 키
- Hugging Face 액세스 토큰

## 1단계: 라이브러리 설치, 데이터 로딩과 준비


구현 코드에서 사용하는 도구와 라이브러리를 간단히 설명하면 다음과 같습니다.
- anthropic: 최신 언어 모델에 접근할 수 있게 해 주는 Anthropic 공식 Python 라이브러리입니다. 텍스트와 이미지를 이해할 수 있는 Claude 3 계열 모델에 접근할 수 있습니다.
- datasets: Hugging Face 생태계의 일부입니다. 'datasets'를 설치하면 전처리가 끝나 바로 사용할 수 있는 여러 데이터셋에 접근할 수 있는데, 이는 머신러닝 모델을 학습·파인튜닝하거나 성능을 벤치마킹하는 데 필수적입니다.
- pandas: 데이터 조작, 처리, 분석을 위한 견고한 자료구조와 메서드를 제공하는 데이터 과학 라이브러리입니다.
- voyageai: VoyageAI의 임베딩 모델군에 접근하기 위한 공식 Python 클라이언트 라이브러리입니다.
- pymongo: MongoDB용 Python 툴킷입니다. MongoDB 데이터베이스와 상호작용할 수 있게 해 줍니다.

In [ ]:
!pip install pymongo datasets pandas anthropic voyageai

아래 코드는 다음 단계를 수행합니다.
1. 필요한 라이브러리 임포트:
- 운영체제와 상호작용하는 `os`,
- HTTP 요청을 보내는 `requests`,
- 메모리상의 파일처럼 바이트 객체를 다루기 위한 io 모듈의 `BytesIO`,
- 데이터 조작과 분석을 위한 `pandas`(pd로 임포트),
- Google Colab 시크릿에 저장된 환경 변수에 접근할 수 있게 해 주는 google.colab의 `userdata`.
2. 함수 정의: `download_and_combine_parquet_files` 함수는 두 개의 파라미터로 정의됩니다.
- `parquet_file_urls`: 문자열 URL의 리스트로, 각각 tech-news-embedding 데이터셋의 일부를 담은 Parquet 파일을 가리킵니다.
- `hf_token`: Hugging Face 인증 토큰을 나타내는 문자열입니다. 액세스 토큰은 [Hugging Face 플랫폼](https://huggingface.co/docs/hub/en/security-tokens#:~:text=To%20create%20an%20access%20token,you%27re%20ready%20to%20go!)에서 만들거나 복사할 수 있습니다.
3. Parquet 파일 내려받기와 읽기: 함수는 parquet_file_urls의 각 URL을 순회하며 다음을 수행합니다.
- requests.get 메서드로 URL과 인증 헤더를 전달해 GET 요청을 보냅니다.
- 응답 상태 코드가 200(OK)인지 확인해 요청이 성공했는지 판단합니다.
- 성공했다면 응답 내용을 BytesIO 객체로 읽어(메모리상의 파일처럼 다루기 위해), pandas.read_parquet으로 이 객체에서 Parquet 파일을 읽어 Pandas DataFrame으로 만듭니다.
- 그 DataFrame을 all_dataframes 리스트에 추가합니다.
4. DataFrame 결합: 모든 Parquet 파일을 내려받아 DataFrame으로 읽은 뒤, `all_dataframes`가 비어 있지 않은지 확인합니다. 다룰 DataFrame이 있으면 pd.concat으로 모두 하나의 DataFrame으로 이어 붙이며, ignore_index=True로 새 결합 DataFrame의 인덱스를 다시 매깁니다. 이 결합 DataFrame이 `download_and_combine_parquet_files` 함수의 최종 출력입니다.

In [ ]:
from io import BytesIO

import pandas as pd
import requests
from google.colab import userdata


def download_and_combine_parquet_files(parquet_file_urls, hf_token):
    """
    Downloads Parquet files from the provided URLs using the given Hugging Face token,
    and returns a combined DataFrame.

    Parameters:
    - parquet_file_urls: List of strings, URLs to the Parquet files.
    - hf_token: String, Hugging Face authorization token.

    Returns:
    - combined_df: A pandas DataFrame containing the combined data from all Parquet files.
    """
    headers = {"Authorization": f"Bearer {hf_token}"}
    all_dataframes = []

    for parquet_file_url in parquet_file_urls:
        response = requests.get(parquet_file_url, headers=headers, timeout=60)
        if response.status_code == 200:
            parquet_bytes = BytesIO(response.content)
            df = pd.read_parquet(parquet_bytes)
            all_dataframes.append(df)
        else:
            print(
                f"Failed to download Parquet file from {parquet_file_url}: {response.status_code}"
            )

    if all_dataframes:
        combined_df = pd.concat(all_dataframes, ignore_index=True)
        return combined_df
    else:
        print("No dataframes to concatenate.")
        return None

아래는 이 튜토리얼에 필요한 Parquet 파일 목록입니다. 전체 파일 목록은 [여기](https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train)에 있습니다. Parquet 파일 하나당 약 45,000개의 데이터 포인트를 담고 있습니다.

아래 코드에서는 tech-news-embeddings 데이터셋의 일부를 하나의 DataFrame으로 묶어 `combined_df` 변수에 할당합니다.

In [3]:
# Uncomment the links below to load more data
# For the full list of data visit: https://huggingface.co/datasets/MongoDB/tech-news-embeddings/tree/refs%2Fconvert%2Fparquet/default/train
parquet_files = [
    "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0000.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0001.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0002.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0003.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0004.parquet",
    # "https://huggingface.co/api/datasets/AIatMongoDB/tech-news-embeddings/parquet/default/train/0005.parquet",
]

hf_token = userdata.get("HF_TOKEN")
combined_df = download_and_combine_parquet_files(parquet_files, hf_token)

데이터 준비의 마지막 단계로, 아래 코드는 묶인 데이터셋에서 `_id` 열을 제거합니다. 이 튜토리얼의 이후 단계에서는 필요하지 않기 때문입니다. 또한 데이터 적재 시 자료형 비호환으로 인한 오류를 막기 위해, 각 데이터 포인트의 embedding 열에 담긴 데이터를 numpy 배열에서 Python 리스트로 변환합니다.

In [4]:
# Remove the _id coloum from the intital dataset
combined_df = combined_df.drop(columns=["_id"])

# Remove the initial embedding coloumn as we are going to create new embeddings with VoyageAI embedding model
combined_df = combined_df.drop(columns=["embedding"])

In [ ]:
combined_df.head()

In [ ]:
# Limiting the amount of document used to 500 for this demo due to the rate limit on VoyageAI API
# Read more on VoyageAI rate limits: https://docs.voyageai.com/docs/rate-limits
max_documents = 500

if len(combined_df) > max_documents:
    combined_df = combined_df[:max_documents]

In [ ]:
import voyageai

vo = voyageai.Client(api_key=userdata.get("VOYAGE_API_KEY"))


def get_embedding(text: str) -> list[float]:
    if not text.strip():
        print("Attempted to get embedding for empty text.")
        return []

    embedding = vo.embed(text, model="voyage-large-2", input_type="document")

    return embedding.embeddings[0]


combined_df["embedding"] = combined_df["description"].apply(get_embedding)

combined_df.head()

## 2단계: 데이터베이스와 컬렉션 생성

**새 MongoDB 데이터베이스를 만들려면 데이터베이스 클러스터를 준비하세요.**
1. [무료 MongoDB Atlas 계정](https://www.mongodb.com/cloud/atlas/register?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)에 가입하거나, 기존 사용자라면 [MongoDB Atlas에 로그인](https://account.mongodb.com/account/login?utm_campaign=devrel&utm_source=community&utm_medium=cta&utm_content=Partner%20Cookbook&utm_term=richmond.alake)하세요.
2. 왼쪽 창에서 “Database” 항목을 선택하면 기존 클러스터의 배포 사양이 표시된 Database Deployment 페이지로 이동합니다. "+Create" 버튼을 눌러 새 데이터베이스 클러스터를 만드세요.
3. 데이터베이스 클러스터 설정과 URI 확보에 도움이 필요하면 MongoDB 클러스터 설정 및 연결 문자열 확보 가이드를 참고하세요.
참고: 개념 검증 단계에서는 Python 호스트의 IP를, 또는 모든 IP를 허용하려면 0.0.0.0/0을 화이트리스트에 등록하는 것을 잊지 마세요.
4. 클러스터가 성공적으로 생성·배포되면 ‘Database Deployment’ 페이지에서 접근할 수 있습니다.
5. 클러스터의 “Connect” 버튼을 누르면 여러 언어 드라이버로 클러스터에 연결하는 방법을 볼 수 있습니다.
6. 이 튜토리얼에는 클러스터의 URI(고유 리소스 식별자)만 필요합니다. URI를 복사해 Google Colab Secrets 환경에 MONGO_URI라는 변수로 넣거나 .env 파일 등에 저장하세요.


클러스터를 만들었다면 클러스터 페이지로 이동해 + Create Database를 눌러 MongoDB Atlas 클러스터 안에 데이터베이스와 컬렉션을 만드세요.
데이터베이스 이름은 `tech_news`, 컬렉션 이름은 `hacker_noon_tech_news`로 합니다.

## 3단계: 벡터 검색 인덱스 생성

여기까지 클러스터, 데이터베이스, 컬렉션을 만들었습니다.

이 절의 단계들은 챗봇에 입력된 질의로 벡터 검색을 수행해 hacker_noon_tech_news 컬렉션의 레코드와 대조할 수 있도록 하는 데 필수적입니다. 목표는 벡터 검색 인덱스를 만드는 것입니다. 이를 위해 공식 [벡터 검색 인덱스 생성 가이드](https://www.mongodb.com/docs/atlas/atlas-vector-search/create-index/)를 참고하세요.

MongoDB Atlas의 JSON 편집기로 벡터 검색 인덱스를 만들 때, 인덱스 이름을 vector_index로 하고 인덱스 정의를 다음과 같이 지정하세요:

```
{
 "fields": [{
     "numDimensions": 1536,
     "path": "embedding",
     "similarity": "cosine",
     "type": "vector"
   }]
}

```

## 4단계: 데이터 적재

앞선 단계에서 만든 MongoDB 데이터베이스에 데이터를 적재하려면 다음 작업을 수행해야 합니다.
- 데이터베이스와 컬렉션에 연결
- 컬렉션의 기존 레코드를 모두 비우기
- 적재 전에 데이터셋의 Pandas DataFrame을 딕셔너리로 변환
- 배치 작업으로 딕셔너리를 MongoDB에 적재

이 튜토리얼에는 클러스터의 URI(고유 리소스 식별자)가 필요합니다. URI를 복사해 Google Colab Secrets 환경에 MONGO_URI라는 변수로 넣거나 .env 파일 등에 저장하세요.

In [6]:
import pymongo
from google.colab import userdata


def get_mongo_client(mongo_uri):
    """Establish connection to the MongoDB."""
    try:
        client = pymongo.MongoClient(mongo_uri)
        print("Connection to MongoDB successful")
        return client
    except pymongo.errors.ConnectionFailure as e:
        print(f"Connection failed: {e}")
        return None


mongo_uri = userdata.get("MONGO_URI")
if not mongo_uri:
    print("MONGO_URI not set in environment variables")

mongo_client = get_mongo_client(mongo_uri)

DB_NAME = "tech_news"
COLLECTION_NAME = "hacker_noon_tech_news"

db = mongo_client[DB_NAME]
collection = db[COLLECTION_NAME]

Connection to MongoDB successful


In [7]:
# To ensure we are working with a fresh collection
# delete any existing records in the collection
collection.delete_many({})

DeleteResult({'n': 228012, 'electionId': ObjectId('7fffffff000000000000000e'), 'opTime': {'ts': Timestamp(1709660559, 7341), 't': 14}, 'ok': 1.0, '$clusterTime': {'clusterTime': Timestamp(1709660559, 7341), 'signature': {'hash': b'jT\xf1\xb4\xa9\xd3\xe3suu\x03`\x15(}\x8f\x00\x9f\xe9\x8a', 'keyId': 7320226449804230661}}, 'operationTime': Timestamp(1709660559, 7341)}, acknowledged=True)

In [ ]:
# Data Ingestion
combined_df_json = combined_df.to_dict(orient="records")
collection.insert_many(combined_df_json)

## 5단계: 벡터 검색

이 절에서는 사용자 질의(챗봇에 입력되는 내용에 해당)를 받는 벡터 검색 커스텀 함수를 만듭니다. 이 함수는 두 번째 파라미터로 `collection`도 받는데, 벡터 검색을 수행할 대상 레코드가 담긴 데이터베이스 컬렉션을 가리킵니다.

`vector_search` 함수는 MongoDB 집계 파이프라인에 정의된 일련의 연산으로 벡터 검색 결과를 만들어 냅니다. 이 파이프라인은 `$vectorSearch`와 `$project` 단계를 포함하며, 사용자 질의의 벡터 임베딩을 기준으로 질의를 수행합니다. 그런 다음 이후 처리에 불필요한 레코드 속성은 빼고 결과를 정리합니다.

아래 코드는 영화 시맨틱 검색을 위해 다음 작업을 수행합니다.
1. 사용자의 질의 문자열과 MongoDB 컬렉션을 입력으로 받아, 벡터 유사도 검색으로 질의에 맞는 문서 목록을 반환하는 `vector_search` 함수를 정의합니다.
2. 앞서 정의한 `get_embedding` 함수를 호출해 사용자 질의의 임베딩을 생성합니다. 이 함수는 질의 문자열을 벡터 표현으로 변환합니다.
3. MongoDB의 aggregate 함수를 위한 파이프라인을 구성하며, `$vectorSearch`와 `$project` 두 주요 단계를 포함합니다.
4. `$vectorSearch` 단계가 실제 벡터 검색을 수행합니다. index 필드는 벡터 검색에 사용할 벡터 인덱스를 지정하며, 앞선 단계의 벡터 검색 인덱스 정의에서 입력한 이름과 일치해야 합니다. queryVector 필드는 사용자 질의의 임베딩 표현을 받습니다. path 필드는 임베딩이 담긴 문서 필드에 해당합니다. `numCandidates`는 고려할 후보 문서 수를, limit은 반환할 결과 수를 지정합니다.
5. $project 단계는 결과에서 _id와 `embedding` 필드를 제외하도록 정리합니다.
6. aggregate가 정의된 파이프라인을 실행해 벡터 검색 결과를 얻습니다. 마지막 연산은 데이터베이스가 반환한 커서를 리스트로 변환합니다.

In [9]:
def vector_search(user_query, collection):
    """
    Perform a vector search in the MongoDB collection based on the user query.

    Args:
    user_query (str): The user's query string.
    collection (MongoCollection): The MongoDB collection to search.

    Returns:
    list: A list of matching documents.
    """

    # Generate embedding for the user query
    query_embedding = get_embedding(user_query)

    if query_embedding is None:
        return "Invalid query or embedding generation failed."

    # Define the vector search pipeline
    pipeline = [
        {
            "$vectorSearch": {
                "index": "vector_index",
                "queryVector": query_embedding,
                "path": "embedding",
                "numCandidates": 150,  # Number of candidate matches to consider
                "limit": 5,  # Return top 5 matches
            }
        },
        {
            "$project": {
                "_id": 0,  # Exclude the _id field
                "embedding": 0,  # Exclude the embedding field
                "score": {
                    "$meta": "vectorSearchScore"  # Include the search score
                },
            }
        },
    ]

    # Execute the search
    results = collection.aggregate(pipeline)
    return list(results)

## 6단계: Claude 3 모델로 사용자 질의 처리하기

튜토리얼의 마지막 절에서는 다음 순서의 작업을 수행합니다.

- 문자열 형태의 사용자 질의를 받습니다.
- VoyageAI 임베딩 모델로 사용자 질의의 임베딩을 생성합니다.
- RAG 시스템의 기반 모델로 Anthropic Claude 3, 구체적으로 ‘claude-opus-4-1’ 모델을 불러옵니다.
- 사용자 질의의 임베딩으로 벡터 검색을 실행해 지식 베이스에서 관련 정보를 가져오고, 이를 기반 모델에 추가 맥락으로 제공합니다.
- 사용자 질의와 수집한 추가 정보를 함께 기반 모델에 보내 응답을 생성합니다.


한 가지 중요한 점은, 사용자 질의 임베딩의 차원이 MongoDB Atlas의 벡터 검색 인덱스 정의에서 설정한 차원과 일치해야 한다는 것입니다.

다음 단계는 anthropic 라이브러리를 임포트하고 클라이언트를 불러와, 메시지를 처리하고 Claude 모델에 접근하는 anthropic의 메서드를 사용하는 것입니다. [Anthropic 공식 사이트](https://console.anthropic.com/settings/keys)의 설정 페이지에서 Claude API 키를 발급받으세요.

In [11]:
import anthropic

client = anthropic.Client(api_key=userdata.get("ANTHROPIC_API_KEY"))

아래 코드의 동작을 더 자세히 설명하면 다음과 같습니다.

1. 벡터 검색 실행: 함수는 사용자 질의와 지정된 컬렉션을 인자로 `vector_search`를 호출하는 것으로 시작합니다. 이는 벡터 임베딩을 활용해 질의와 관련된 정보를 컬렉션 안에서 찾습니다.
2. 검색 결과 정리: `search_result`를 빈 문자열로 초기화해 검색 정보를 모읍니다. `vector_search` 함수가 반환한 결과를 순회하며 각 항목의 세부 정보(제목, 회사명, URL, 게시일, 기사 URL, 설명)를 사람이 읽을 수 있는 문자열로 정리하고, 각 항목 끝에 개행 문자 \n을 붙여 search_result에 이어 붙입니다.
3. Anthropic 클라이언트로 응답 생성: 그런 다음 함수는 Claude API에 보낼 요청을 구성합니다(앞서 만든 anthropic.Client 클래스의 인스턴스로 보이는 client 객체를 통해). 다음을 지정합니다.
- 사용할 모델("claude-opus-4-1")로, Claude 3 모델의 특정 버전을 가리킵니다.
- 생성 응답의 최대 토큰 한도(max_tokens=1024).
- 기술 기업 기사와 정보에 접근할 수 있는 "벤처캐피털 기술 애널리스트"처럼 행동하도록 모델을 안내하는 시스템 설명. 이 맥락을 활용해 조언합니다.
- 모델이 처리할 실제 메시지로, 사용자 질의와 정리된 검색 결과를 맥락으로 결합한 것입니다.
4. 생성된 응답과 검색 결과 반환: 응답 content의 첫 항목에서 응답 텍스트를 추출해, 정리된 검색 결과와 함께 반환합니다.

In [12]:
def handle_user_query(query, collection):
    get_knowledge = vector_search(query, collection)

    search_result = ""
    for result in get_knowledge:
        search_result += (
            f"Title: {result.get('title', 'N/A')}, "
            f"Company Name: {result.get('companyName', 'N/A')}, "
            f"Company URL: {result.get('companyUrl', 'N/A')}, "
            f"Date Published: {result.get('published_at', 'N/A')}, "
            f"Article URL: {result.get('url', 'N/A')}, "
            f"Description: {result.get('description', 'N/A')}, \n"
        )

    response = client.messages.create(
        model="claude-opus-4-1",
        max_tokens=1024,
        system="You are Venture Captital Tech Analyst with access to some tech company articles and information. You use the information you are given to provide advice.",
        messages=[
            {
                "role": "user",
                "content": "Answer this user query: "
                + query
                + " with the following context: "
                + search_result,
            }
        ],
    )

    return (response.content[0].text), search_result

이 튜토리얼의 마지막 단계는 질의를 초기화해 `handle_user_query` 함수에 전달하고, 반환된 응답을 출력하는 것입니다.

In [13]:
# Conduct query with retrieval of sources
query = "Give me the best tech stock to invest in and tell me why"
response, source_information = handle_user_query(query, collection)

print(f"Response: {response}")
print(f"\\nSource Information: \\n{source_information}")

Response: Based on the information provided in the article titles and descriptions, Alibaba Group Holding Limited appears to be a top technology stock pick for 2023 according to renowned investor Ray Dalio. The article "Top 10 Technology Stocks to Buy in 2023 According to Ray Dalio" suggests that Alibaba is one of Dalio's favored tech investments for the year.

As a venture capital tech analyst, I would recommend considering an investment in Alibaba for the following reasons:

1. Endorsement from a respected investor: Ray Dalio, known for his successful investment strategies, has included Alibaba in his top 10 technology stock picks for 2023. His backing lends credibility to the investment potential of the company.

2. Strong market position: Alibaba is a leading e-commerce company in China with a significant market share. It has a diversified business model spanning e-commerce, cloud computing, digital media, and entertainment.

3. Growth potential: With China's large and growing midd